In [ ]:
if (!requireNamespace("flextable", quietly = TRUE)) {
  install.packages("flextable")
}

if (!requireNamespace("officer", quietly = TRUE)) {
  install.packages("officer")
}

if (!requireNamespace("janitor", quietly = TRUE)) {
  install.packages("janitor")
}

if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install("biomaRt")

install.packages("xlsx")

install.packages("paletteer")

In [ ]:
# Load the package
library(tidyverse)
library(ggplot2)
library(data.table)
library(janitor)
library(flextable)
library(officer)
library(xlsx)
library(biomaRt)

In [ ]:
# Connect to Ensembl
mart <- useMart("ensembl", dataset = "hsapiens_gene_ensembl")

In [ ]:

# Directory containing gene set-based rare variant assoication test results 
files <- list.files(path="merged_saige_gene_step02")
# Load file containing conversion from Ensembl gene IDs to HUGO gene symbols file 
# gene_symbols <- rename(fread("gene_symbols.tsv"), "Region" = "gene_id")

# Function: Formats the gene set-based rare variant assoication test results 
process_results <- function(results) { 
    results %>% 
    filter(Group != "Cauchy") %>% 
    # inner_join(gene_symbols, by = "Region") %>% 
    mutate(max_MAF = as.character(max_MAF)) %>% 
    mutate(negative_log_p_values = -log10(Pvalue)) %>% 
    mutate(Chromosome = gsub("chr", "", Chromosome)) %>% 
    mutate(Chromosome = as.numeric(Chromosome)) %>% 
    mutate(Region = as.factor(Region)) %>% 
    mutate(Region_factor = fct_reorder(Region, Chromosome))
  }

# Loop: Loads and format files 
for (f in files) {
  print(f)
  data <- fread(paste0("merged_saige_gene_step02/", f))
  processed <- process_results(data)
    
  # Loop through unique Ensembl IDs
  gene_info <- getBM(
    attributes = c("ensembl_gene_id", "hgnc_symbol"),
    filters = "ensembl_gene_id",
    values = unique(processed$Region),
    mart = mart
  )
  # Rename columns
  gene_info <- 
  rename(gene_info, 
       "Region" = "ensembl_gene_id", 
       "Symbol" = "hgnc_symbol" )
  # Join together Symbols with P-values 
  processed <- full_join(gene_info, processed, by ="Region")
  # Assign to the variable 
  assign(f, processed, envir = .GlobalEnv)
}

In [ ]:
# Create a list of gene set-based rare variant assoication test results 
df_list <- list("eur_aml_phenotype_high_confidence" = EUR_aml_phenotype_high_confidence_merged_chr,
                "eur_aml_phenotype_high_confidence_deprivation_index" = EUR_aml_phenotype_high_confidence_merged_chr_deprivation_index,
                "eur_aml_phenotype_synonymous" = EUR_aml_phenotype_synonymous_merged_chr,
                "eur_aml_phenotype_synonymous_deprivation_index" = EUR_aml_phenotype_synonymous_merged_chr_deprivation_index)

In [ ]:
genomic_inflation_list <- list()
for (df_name in names(df_list)) {
    df <- df_list[[df_name]]  # get the current data frame
    # Transform the full dataset for plotting
    
    genomic_inflation <- 
    df %>%
    filter(!is.na(Pvalue) & Pvalue > 0 & Pvalue <= 1) %>%
    pivot_longer(
    cols = starts_with("Pvalue"),
    names_to = "Statistical_Test",
    names_prefix = "Pvalue_",
    values_to = "P_value",
    values_drop_na = TRUE
    ) %>%
     mutate(
         Statistical_Test = ifelse(Statistical_Test == "Pvalue", "SKAT-O", Statistical_Test)
     ) %>%
    mutate(max_MAF = case_when(
    max_MAF == "0.01"   ~ "1 x 10⁻²",
    max_MAF == "0.001"  ~ "1 x 10⁻³",
    max_MAF == "1e-04"  ~ "1 x 10⁻⁴",
    TRUE ~ max_MAF)) %>%
    group_by(max_MAF, Statistical_Test) %>% 
    mutate(chisq = qchisq(P_value,1,lower.tail=FALSE)) %>% 
    group_by(max_MAF, Statistical_Test) %>% 
    mutate(lambda = (median(chisq) / qchisq(0.5,1)),
           lambda_0.01 = quantile(chisq, probs = 0.99) / qchisq(0.99, df = 1)) %>% 
    distinct(Statistical_Test,max_MAF,lambda,lambda_0.01) %>% 
    rename("Statistical Test" = "Statistical_Test", 
         "Maximum MAF" = "max_MAF", 
         "Lambda" = "lambda",
         "Lambda 0.01" = "lambda_0.01") 

    genomic_inflation$Analysis <- df_name
    genomic_inflation_list[[df_name]] <- genomic_inflation

}


In [ ]:
genomic_inflation_df <- do.call(rbind, genomic_inflation_list)

In [ ]:
genomic_inflation_df <- genomic_inflation_df[, c("Analysis", "Statistical Test", 
                                                 "Maximum MAF", "Lambda", "Lambda 0.01")]


In [ ]:
# Write Excel
write.xlsx(as.data.frame(genomic_inflation_df), "genomic_inflation.xlsx")

In [ ]:
options(repr.plot.height = 15, repr.plot.width = 15)

In [ ]:
for (df_name in names(df_list)) {
  df <- df_list[[df_name]]  # get the current data frame
# Transform the full dataset for plotting

# Transform the full dataset for plotting
plot_data <- df %>%
  filter(!is.na(Pvalue) & Pvalue > 0 & Pvalue <= 1) %>%
  mutate(max_MAF = case_when(
    max_MAF == "0.01"  ~ "1 x 10⁻²",
    max_MAF == "0.001" ~ "1 x 10⁻³",
    max_MAF == "1e-04"  ~ "1 x 10⁻⁴",
    TRUE ~ max_MAF  # keep other values unchanged
  )) %>% 
  pivot_longer(
    cols = starts_with("Pvalue"),
    names_to = "Statistical_Test",
    names_prefix = "Pvalue_",
    values_to = "P_value",
    values_drop_na = TRUE
  ) %>%
  mutate(
    Statistical_Test = ifelse(Statistical_Test == "Pvalue", "SKAT-O", Statistical_Test)
  ) %>%
  group_by(max_MAF, Statistical_Test) %>%
  arrange(P_value, .by_group = TRUE) %>%
  mutate(
    observed = -log10(P_value),
    expected = -log10(ppoints(n()))
  ) %>%
  ungroup()

# Get the top 10 rows based on smallest P_value from the plot data
top_values <- plot_data %>%
  arrange(P_value) %>%
  slice(1:10)

# Final QQ plot with labels placed at correct expected positions
plot <- ggplot(plot_data, aes(x = expected, y = observed)) +
  geom_point(size = 2, aes(color = Statistical_Test)) +
  geom_abline(slope = 1, intercept = 0, color = "black", linetype = "dashed") +
  facet_grid(rows = vars(max_MAF), cols = vars(Statistical_Test), scales = "free") +
  labs(
    x = "Expected -log10(p)",
    y = "Observed -log10(p)"
  ) +
  theme_minimal() + 
  scale_color_manual(values = rev(paletteer::paletteer_d("LaCroixColoR::Lemon"))) + 
  theme(
  axis.title.x = element_text(size = 20),
  axis.title.y = element_text(size = 20),
  legend.title = element_text(size = 16), 
  legend.text = element_text(size = 14),
  strip.text = element_text(size = 20),
  panel.border = element_rect(colour = "black", fill = NA, linewidth = 1.2) # optional: box around entire plot
  ) + 
  guides(size = "none", color =  "none") +
  geom_text(
    data = top_values,
    aes(label = Symbol),
    check_overlap = TRUE,
    vjust = 0.5,
    hjust = 1.1,
    size = 4
  )

  # Save plot
  ggsave(filename = paste0("qq_", df_name, "_plot.png"), 
         plot = plot, 
         width = 15, 
         height = 15,
         dpi = 300)
}

